In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/math482-2024-2025-1-hw-03/sample_submission.csv
/kaggle/input/math482-2024-2025-1-hw-03/train.csv
/kaggle/input/math482-2024-2025-1-hw-03/test.csv


# **1. Import Libraries**

In [2]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression 
from sklearn.svm import SVR
from sklearn.preprocessing import OrdinalEncoder, PolynomialFeatures, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
import tensorflow as tf

# **2. Load the Datasets**

In [3]:
train_df = pd.read_csv('/kaggle/input/math482-2024-2025-1-hw-03/train.csv')
test_df = pd.read_csv('/kaggle/input/math482-2024-2025-1-hw-03/test.csv')
train_df.head(-1)

,id,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,...,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,target
0,1,43.71,-90.85,u,-28.35,0.20,0.24,B1,NaN,14.65,...,2.98,ij,5.29,C3,8.0,A2,ab,D4,654.69,357.98
1,2,95.56,-9.20,NaN,-14.07,0.91,0.04,B3,106.68,2.41,...,0.86,ij,-12.72,C1,6.0,A4,ac,D1,2.83,1103.49
2,3,75.88,NaN,u,-8.50,0.50,0.13,B3,70.79,3.10,...,7.83,ii,-3.47,C2,1.0,A5,ab,D1,312.83,501.38
3,4,63.88,NaN,q,NaN,3.07,0.36,B6,90.18,1.07,...,0.73,ji,31.65,C1,3.0,A8,ad,D1,1.99,1174.88
4,5,24.04,-41.22,t,-32.25,4.86,0.42,B5,119.19,8.07,...,1.12,jj,-12.25,NaN,5.0,A7,NaN,NaN,0.85,1350.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29994,29995,34.24,71.61,NaN,41.57,0.66,0.39,B4,113.34,3.42,...,0.79,ii,33.43,C5,2.0,A7,ae,D1,3.00,1967.36
29995,29996,95.70,NaN,s,7822.04,1.34,0.03,B2,110.81,NaN,...,123.04,ji,-20.34,C3,3.0,A2,ac,D1,1.10,1756.51
29996,29997,NaN,31.39,q,32.65,0.64,0.15,B5,NaN,6.24,...,2.63,ij,NaN,C3,0.0,A7,ab,D4,0.86,2048.25
29997,29998,NaN,-37.90,s,NaN,NaN,0.14,B6,67.71,8.14,...,1.44,ij,411.17,C5,4.0,A5,aa,D1,2.95,508.91


# **3. Preprocessing the Data**

In [4]:
train_df.dtypes

id              int64
feature_01    float64
feature_02    float64
feature_03     object
feature_04    float64
feature_05    float64
feature_06    float64
feature_07     object
feature_08    float64
feature_09    float64
feature_10    float64
feature_11     object
feature_12    float64
feature_13    float64
feature_14     object
feature_15    float64
feature_16     object
feature_17    float64
feature_18     object
feature_19     object
feature_20     object
feature_21    float64
target        float64
dtype: object

In [5]:
test_df.dtypes

id              int64
feature_01    float64
feature_02    float64
feature_03     object
feature_04    float64
feature_05    float64
feature_06    float64
feature_07     object
feature_08    float64
feature_09    float64
feature_10    float64
feature_11     object
feature_12    float64
feature_13    float64
feature_14     object
feature_15    float64
feature_16     object
feature_17    float64
feature_18     object
feature_19     object
feature_20     object
feature_21    float64
dtype: object

# 3.1 : Missing Values

In [6]:
# get the number of missing data points per column
missing_values_count = train_df.isnull().sum()

# look at the # of missing points in the columns
missing_values_count[0:23]

id               0
feature_01    2848
feature_02    2864
feature_03    2805
feature_04    2890
feature_05    2870
feature_06    2842
feature_07    2830
feature_08    2880
feature_09    2875
feature_10    2907
feature_11    2848
feature_12    2884
feature_13    2854
feature_14       0
feature_15    2823
feature_16    2863
feature_17    2847
feature_18    2858
feature_19    2848
feature_20    2833
feature_21    2904
target           0
dtype: int64

In [7]:
# how many total missing values do we have?
total_cells = np.product(train_df.shape)
total_missing = missing_values_count.sum()

# percent of data that is missing
percent_missing = (total_missing/total_cells) * 100
print(percent_missing)

8.285942028985508


Almost all columns in the training dataset have missing values. So it is not effecting way to drops the columns or rows with missing values. Thus, it is better to use imputation.

In [8]:
# replace all NA's the value that comes directly after it in the same column, 
# then replace all the remaining na's with 0

train_df = train_df.bfill().fillna(0)
train_df.head()


,id,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,...,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,target
0,1,43.71,-90.85,u,-28.35,0.20,0.24,B1,106.68,14.65,...,2.98,ij,5.29,C3,8.0,A2,ab,D4,654.69,357.98
1,2,95.56,-9.20,u,-14.07,0.91,0.04,B3,106.68,2.41,...,0.86,ij,-12.72,C1,6.0,A4,ac,D1,2.83,1103.49
2,3,75.88,-41.22,u,-8.50,0.50,0.13,B3,70.79,3.10,...,7.83,ii,-3.47,C2,1.0,A5,ab,D1,312.83,501.38
3,4,63.88,-41.22,q,-32.25,3.07,0.36,B6,90.18,1.07,...,0.73,ji,31.65,C1,3.0,A8,ad,D1,1.99,1174.88
4,5,24.04,-41.22,t,-32.25,4.86,0.42,B5,119.19,8.07,...,1.12,jj,-12.25,C4,5.0,A7,ad,D3,0.85,1350.99


In [9]:
test_df = test_df.bfill().fillna(0)
train_df.head()

,id,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,...,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,target
0,1,43.71,-90.85,u,-28.35,0.20,0.24,B1,106.68,14.65,...,2.98,ij,5.29,C3,8.0,A2,ab,D4,654.69,357.98
1,2,95.56,-9.20,u,-14.07,0.91,0.04,B3,106.68,2.41,...,0.86,ij,-12.72,C1,6.0,A4,ac,D1,2.83,1103.49
2,3,75.88,-41.22,u,-8.50,0.50,0.13,B3,70.79,3.10,...,7.83,ii,-3.47,C2,1.0,A5,ab,D1,312.83,501.38
3,4,63.88,-41.22,q,-32.25,3.07,0.36,B6,90.18,1.07,...,0.73,ji,31.65,C1,3.0,A8,ad,D1,1.99,1174.88
4,5,24.04,-41.22,t,-32.25,4.86,0.42,B5,119.19,8.07,...,1.12,jj,-12.25,C4,5.0,A7,ad,D3,0.85,1350.99


In [10]:
# get the number of missing data points per column
missing_values_count = train_df.isnull().sum()

# look at the # of missing points in the first ten columns
missing_values_count[0:23]

id            0
feature_01    0
feature_02    0
feature_03    0
feature_04    0
feature_05    0
feature_06    0
feature_07    0
feature_08    0
feature_09    0
feature_10    0
feature_11    0
feature_12    0
feature_13    0
feature_14    0
feature_15    0
feature_16    0
feature_17    0
feature_18    0
feature_19    0
feature_20    0
feature_21    0
target        0
dtype: int64

# 3.2: Handle Categorical Variables

In [11]:
# Get list of categorical variables
s = (train_df.dtypes == 'object')
object_cols = list(s[s].index)

print("Categorical variables:")
print(object_cols)

Categorical variables:
['feature_03', 'feature_07', 'feature_11', 'feature_14', 'feature_16', 'feature_18', 'feature_19', 'feature_20']


In [12]:
for col in object_cols:
    unique_values = train_df[col].unique()
    print(f"Unique values in '{col}': {unique_values}")

Unique values in 'feature_03': ['u' 'q' 't' 'r' 's' 'p']
Unique values in 'feature_07': ['B1' 'B3' 'B6' 'B5' 'B4' 'B2']
Unique values in 'feature_11': ['xy' 'yy' 'xx' 'yx']
Unique values in 'feature_14': ['ij' 'ii' 'ji' 'jj']
Unique values in 'feature_16': ['C3' 'C1' 'C2' 'C4' 'C5']
Unique values in 'feature_18': ['A2' 'A4' 'A5' 'A8' 'A7' 'A1' 'A3' 'A6']
Unique values in 'feature_19': ['ab' 'ac' 'ad' 'ae' 'aa' 0]
Unique values in 'feature_20': ['D4' 'D1' 'D3' 'D5' 'D6' 'D2']


First we convert the ordinal values into numerical values using ordinal encoding. **The Ordinal Encoder** converts categorical data into integers based on the order of the categories, making it suitable for variables with an inherent ranking (e.g., "Low," "Medium," "High"). It assigns a unique integer to each category while preserving the logical order (e.g., "Low" → 0, "Medium" → 1, "High" → 2). This encoding allows machine learning models to understand the ordinal relationship between categories.

In [13]:
train_df['feature_19'] = train_df['feature_19'].replace(0, 'aa')

# Define the ordinal categories for each feature
ordinals = {
    'feature_07': ['B1', 'B2', 'B3', 'B4', 'B5', 'B6'],
    'feature_16': ['C1', 'C2', 'C3', 'C4', 'C5'],
    'feature_18': ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8'],
    'feature_20': ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']
}

encoder = OrdinalEncoder(categories=[ordinals[feature] for feature in ordinals.keys()])

# Fit and transform the specified features

train_df[list(ordinals.keys())] = encoder.fit_transform(train_df[list(ordinals.keys())])

# Display the transformed DataFrame
train_df.head(-1)

,id,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,...,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,target
0,1,43.71,-90.85,u,-28.35,0.20,0.24,0.0,106.68,14.65,...,2.98,ij,5.29,2.0,8.0,1.0,ab,3.0,654.69,357.98
1,2,95.56,-9.20,u,-14.07,0.91,0.04,2.0,106.68,2.41,...,0.86,ij,-12.72,0.0,6.0,3.0,ac,0.0,2.83,1103.49
2,3,75.88,-41.22,u,-8.50,0.50,0.13,2.0,70.79,3.10,...,7.83,ii,-3.47,1.0,1.0,4.0,ab,0.0,312.83,501.38
3,4,63.88,-41.22,q,-32.25,3.07,0.36,5.0,90.18,1.07,...,0.73,ji,31.65,0.0,3.0,7.0,ad,0.0,1.99,1174.88
4,5,24.04,-41.22,t,-32.25,4.86,0.42,4.0,119.19,8.07,...,1.12,jj,-12.25,3.0,5.0,6.0,ad,2.0,0.85,1350.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29994,29995,34.24,71.61,s,41.57,0.66,0.39,3.0,113.34,3.42,...,0.79,ii,33.43,4.0,2.0,6.0,ae,0.0,3.00,1967.36
29995,29996,95.70,31.39,s,7822.04,1.34,0.03,1.0,110.81,6.24,...,123.04,ji,-20.34,2.0,3.0,1.0,ac,0.0,1.10,1756.51
29996,29997,46.09,31.39,q,32.65,0.64,0.15,4.0,67.71,6.24,...,2.63,ij,411.17,2.0,0.0,6.0,ab,3.0,0.86,2048.25
29997,29998,46.09,-37.90,s,-36.70,1.23,0.14,5.0,67.71,8.14,...,1.44,ij,411.17,4.0,4.0,4.0,aa,0.0,2.95,508.91


In [14]:
for col in object_cols:
    unique_values_2 = test_df[col].unique()
    print(f"Unique values in '{col}': {unique_values_2}")

Unique values in 'feature_03': ['t' 's' 'u' 'r' 'p' 'q']
Unique values in 'feature_07': ['B5' 'B4' 'B1' 'B6' 'B3' 'B2']
Unique values in 'feature_11': ['yy' 'xx' 'yx' 'xy' 0]
Unique values in 'feature_14': ['ii' 'ji' 'ij' 'jj']
Unique values in 'feature_16': ['C5' 'C1' 'C4' 'C2' 'C3']
Unique values in 'feature_18': ['A2' 'A3' 'A7' 'A4' 'A5' 'A8' 'A1' 'A6']
Unique values in 'feature_19': ['ab' 'ac' 'ad' 'aa' 'ae']
Unique values in 'feature_20': ['D1' 'D3' 'D2' 'D4' 'D5' 'D6']


In [15]:
test_df['feature_11'] = test_df['feature_11'].replace(0, 'xx')
# test has the same ordinals

encoder = OrdinalEncoder(categories=[ordinals[feature] for feature in ordinals.keys()])

# Fit and transform the specified features

test_df[list(ordinals.keys())] = encoder.fit_transform(test_df[list(ordinals.keys())])

# Display the transformed DataFrame
test_df.head(-1)

,id,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,...,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21
0,30001,67.43,-60.77,t,89.21,379.47,0.21,4.0,91.59,0.11,...,1.56,0.46,ii,-24.21,4.0,3.0,1.0,ab,0.0,9.71
1,30002,51.34,13.91,s,89.21,2.76,0.22,3.0,91.59,18.08,...,1.47,0.20,ji,47.39,0.0,3.0,2.0,ac,0.0,6.61
2,30003,96.80,-44.69,u,30.18,1.86,0.41,0.0,85.75,21.54,...,1.40,0.20,ji,-45.33,0.0,8.0,6.0,ab,2.0,9.77
3,30004,29.71,-130.91,t,-19.33,1.48,0.40,3.0,62.30,28.85,...,43.18,0.62,ji,20.79,0.0,4.0,6.0,ab,1.0,282.67
4,30005,62.91,38.34,u,-15.25,1.72,0.42,3.0,49.90,5.99,...,16.71,0.40,ij,-38.91,3.0,4.0,3.0,ad,3.0,4.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9994,39995,5996.64,20.40,q,23.95,4.16,0.29,4.0,101.77,1.15,...,18.74,1.64,jj,10.26,3.0,5.0,6.0,ab,0.0,4.10
9995,39996,77.86,-33.12,p,-20.40,0.28,0.41,0.0,7951.91,14.14,...,19.99,0.88,ii,-3050.88,0.0,5.0,4.0,ab,0.0,568.70
9996,39997,3157.67,-26.55,p,92.72,0.28,0.25,4.0,82.52,14.14,...,1009.46,0.88,ii,-40.04,0.0,7.0,1.0,ad,5.0,2.88
9997,39998,3157.67,-59.06,u,-54.23,1.41,41.38,0.0,82.15,7.65,...,2.34,1.12,ij,-14.81,1.0,6.0,1.0,ad,5.0,0.62


Now, let's deal with nominal values using Label Encoder. **The Label Encoder** is a technique for converting categorical data into numerical form by assigning a unique integer to each category without considering any order. It maps each category to an integer, enabling machine learning models to process categorical data as numerical inputs. Unlike the Ordinal Encoder, it does not imply or preserve any inherent order among the categories, making it suitable for unordered categorical data.

In [16]:
# Get list of categorical variables
s = (train_df.dtypes == 'object')
object_cols = list(s[s].index)

print("Categorical variables:")
print(object_cols)

Categorical variables:
['feature_03', 'feature_11', 'feature_14', 'feature_19']


In [17]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [18]:
train_df.feature_03 = le.fit_transform(train_df.feature_03)
train_df.feature_11 = le.fit_transform(train_df.feature_11)
train_df.feature_14 = le.fit_transform(train_df.feature_14)
train_df.feature_19 = le.fit_transform(train_df.feature_19)


train_df.head()

,id,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,...,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,target
0,1,43.71,-90.85,5,-28.35,0.20,0.24,0.0,106.68,14.65,...,2.98,1,5.29,2.0,8.0,1.0,1,3.0,654.69,357.98
1,2,95.56,-9.20,5,-14.07,0.91,0.04,2.0,106.68,2.41,...,0.86,1,-12.72,0.0,6.0,3.0,2,0.0,2.83,1103.49
2,3,75.88,-41.22,5,-8.50,0.50,0.13,2.0,70.79,3.10,...,7.83,0,-3.47,1.0,1.0,4.0,1,0.0,312.83,501.38
3,4,63.88,-41.22,1,-32.25,3.07,0.36,5.0,90.18,1.07,...,0.73,2,31.65,0.0,3.0,7.0,3,0.0,1.99,1174.88
4,5,24.04,-41.22,4,-32.25,4.86,0.42,4.0,119.19,8.07,...,1.12,3,-12.25,3.0,5.0,6.0,3,2.0,0.85,1350.99


In [19]:
test_df.feature_03 = le.fit_transform(test_df.feature_03)
test_df.feature_11 = le.fit_transform(test_df.feature_11)
test_df.feature_14 = le.fit_transform(test_df.feature_14)
test_df.feature_19 = le.fit_transform(test_df.feature_19)


test_df.head()

,id,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,...,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21
0,30001,67.43,-60.77,4,89.21,379.47,0.21,4.0,91.59,0.11,...,1.56,0.46,0,-24.21,4.0,3.0,1.0,1,0.0,9.71
1,30002,51.34,13.91,3,89.21,2.76,0.22,3.0,91.59,18.08,...,1.47,0.20,2,47.39,0.0,3.0,2.0,2,0.0,6.61
2,30003,96.80,-44.69,5,30.18,1.86,0.41,0.0,85.75,21.54,...,1.40,0.20,2,-45.33,0.0,8.0,6.0,1,2.0,9.77
3,30004,29.71,-130.91,4,-19.33,1.48,0.40,3.0,62.30,28.85,...,43.18,0.62,2,20.79,0.0,4.0,6.0,1,1.0,282.67
4,30005,62.91,38.34,5,-15.25,1.72,0.42,3.0,49.90,5.99,...,16.71,0.40,1,-38.91,3.0,4.0,3.0,3,3.0,4.06


# 3.3: Identify Irrelevant Features

Now, we identify irrelevent features by Random Forest Regressor method and form the datasets again.
**The Random Forest Regressor** is an ensemble machine learning algorithm that predicts continuous target variables by combining the outputs of multiple decision trees. Each tree is trained on a random subset of the data (using bootstrapping) and considers a random subset of features at each split, ensuring diversity among trees. The final prediction is made by averaging the predictions of all the trees, which reduces variance and improves generalization. This method is robust to overfitting, handles non-linear relationships well, and can work effectively with large datasets.

In [20]:

from sklearn.ensemble import RandomForestRegressor

# Separate target variable
X_train_full = train_df.drop(columns=['target'])
y_train_full = train_df['target']
X_test = test_df.copy()

# Detect and Drop Irrelevant Features
# Use a quick RandomForest model to assess feature importance

forest = RandomForestRegressor(random_state=42)
forest.fit(X_train_full.fillna(0), y_train_full)  # Fill missing values temporarily for this step

# Threshold for importance (example: keep features with > 1% importance)
feature_importances = pd.Series(forest.feature_importances_, index=X_train_full.columns)
important_features = feature_importances[feature_importances > 0.01].index.tolist()

# Keep only important features
X_train = X_train_full[important_features]
X_test = X_test[important_features]
X_train.head()

,id,feature_01,feature_02,feature_04,feature_05,feature_08,feature_09,feature_10,feature_12,feature_13,feature_15,feature_21
0,1,43.71,-90.85,-28.35,0.20,106.68,14.65,30.56,8.63,2.98,5.29,654.69
1,2,95.56,-9.20,-14.07,0.91,106.68,2.41,7512.06,0.58,0.86,-12.72,2.83
2,3,75.88,-41.22,-8.50,0.50,70.79,3.10,13.32,2.67,7.83,-3.47,312.83
3,4,63.88,-41.22,-32.25,3.07,90.18,1.07,82.79,1.49,0.73,31.65,1.99
4,5,24.04,-41.22,-32.25,4.86,119.19,8.07,82.79,10.33,1.12,-12.25,0.85


# 4. Construct Validation Set

THe primary purpose of creating a validation set is to simulate how the model might perform on unseen data, ensuring it generalizes well and avoids overfitting. By assessing performance on the validation set, you can determine the optimal model configuration without relying on the test set, preserving the integrity of the final evaluation.

In [21]:
# Split data into train and validation sets

X = X_train_full
y = y_train_full
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


# 5. Train Model

**Linear Regression** models the relationship as a straight line or hyperplane in higher dimensions by minimizing the residual sum of squares , i.e. the difference between observed and predicted values. The model assumes a linear relationship between the features and the target

In [22]:
# Linear Regression

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_val_pred_lr = lin_reg.predict(X_val)

**Polynomial regression** extends linear regression by fitting a polynomial equation to the data instead of a straight line. It achieves this by introducing polynomial features to the model, allowing it to capture non-linear relationships between the input features and the target variable. Despite the added flexibility, polynomial regression can be prone to overfitting if the polynomial degree is too high, making careful selection of the degree important.

In [23]:
# Polynomial Regression

poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
poly_reg = LinearRegression()
poly_reg.fit(X_train_poly, y_train)
y_val_pred_pr = poly_reg.predict(poly.transform(X_val))

**Support Vector Regression** is a regression variant of Support Vector Machines (SVMs). It aims to find a function that predicts target values within a specified margin of tolerance (epsilon). SVR works by mapping input features to a higher-dimensional space using a kernel function and finding a hyperplane that fits the data with the least error while maintaining the margin. It is robust to outliers and effective for both linear and non-linear relationships, depending on the choice of kernel (e.g., linear, polynomial, or radial basis function). 'rbf' stands for Radial Basis Function, a commonly used kernel that models non-linear relationships. It computes the similarity between data points based on their distance, effectively capturing complex patterns.

In [24]:
# Support Vector Regression
svr = SVR(kernel='rbf', C=1.0) # C=1.0 for moderate regularization to balance training accuracy and generalization.
svr.fit(X_train, y_train)
y_val_pred_svr = svr.predict(X_val)


**Neural networks** are a flexible and powerful machine learning method inspired by the human brain, consisting of interconnected layers of nodes (neurons). Each neuron applies a weighted sum of inputs followed by an activation function to model complex, non-linear relationships between inputs and outputs. Neural networks learn by optimizing weights using backpropagation and gradient descent. 

* **tf.keras.Sequential** is a class in TensorFlow that allows you to build a neural network layer by layer in a sequential manner, where each layer has one input and one output.
* **Dense** is a fully connected layer where every neuron in the current layer is connected to every neuron in the previous layer.
  
  * Parameters:
    * **64, 32, 1**: Number of neurons in each layer.
            The first layer has 64 neurons, the second has 32, and the output layer has 1 neuron, designed for a     regression task.
      
    * **activation='relu'**: The activation function applied to the layer’s output.
relu (Rectified Linear Unit) is a common activation function that introduces non-linearity by outputting 
max(0,𝑥).
The output layer has no activation function (default is linear) as it is a regression task.

* **nn.compile:**
This method configures the model for training by specifying the optimizer, loss function, and evaluation metrics.

    * **optimizer='adam'**: The optimizer determines how weights are updated during backpropagation.
      
    **adam (Adaptive Moment Estimation)** is an efficient and widely used optimizer that adjusts the learning rate dynamically based on the first and second moments of the gradients.

In [25]:
# Neural Network

nn = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)])
nn.compile(optimizer='adam', loss='mse', metrics=['mse'])
nn.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val), batch_size=32)
y_val_pred_nn = nn.predict(X_val)

Epoch 1/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 688288.8750 - mse: 688288.8750 - val_loss: 646227.2500 - val_mse: 646227.2500
Epoch 2/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 635709.3750 - mse: 635709.3750 - val_loss: 582947.3750 - val_mse: 582947.3750
Epoch 3/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 574177.4375 - mse: 574177.4375 - val_loss: 450271.3750 - val_mse: 450271.3750
Epoch 4/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 452733.1562 - mse: 452733.1562 - val_loss: 388755.2500 - val_mse: 388755.2500
Epoch 5/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 388007.7500 - mse: 388007.7500 - val_loss: 373080.1562 - val_mse: 373080.1562
Epoch 6/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 385693.5625 - mse: 385693.5625 - val_loss: 378678.6875 - val_mse: 378678.6875
Epoch 7/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 369750.5938 - mse: 369750.5938 - val_loss: 354676.4062 - val_mse: 354676.4062
Epoch 8/50
750/750 ━━━━━━━━━━━━━━━

# 5. Compare Models

In [26]:

metrics = {
    'Model': ['Linear Regression', 'Polynomial Regression', 'SVR', 'Neural Network'],
    'MSE': [mean_squared_error(y_val, y_val_pred_lr),
            mean_squared_error(y_val, y_val_pred_pr),
            mean_squared_error(y_val, y_val_pred_svr),
            mean_squared_error(y_val, y_val_pred_nn)],
    'R2': [r2_score(y_val, y_val_pred_lr),
           r2_score(y_val, y_val_pred_pr),
           r2_score(y_val, y_val_pred_svr),
           r2_score(y_val, y_val_pred_nn)]}
results_df = pd.DataFrame(metrics)
print(results_df)

                   Model            MSE        R2
0      Linear Regression  430315.125586  0.059701
1  Polynomial Regression  425760.815691  0.069652
2                    SVR  454655.428851  0.006514
3         Neural Network  182820.880291  0.600511


We continue with the neural network method since it gives less error compared to other models.

In [27]:
# Predict on test data with best model and save results

best_model = nn  

pred = best_model.predict(test_df)


313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 713us/step


In [28]:
print("Pred:", pred)

pred= pred.flatten() 
output = pd.DataFrame({'id': X_test.id, 'target': pred})
output.head(-1)


Pred: [[1003.03485]
 [1595.9724 ]
 [ 364.64478]
 ...
 [ 228.54018]
 [1191.7161 ]
 [1440.3269 ]]


,id,target
0,30001,1003.034851
1,30002,1595.972412
2,30003,364.644775
3,30004,605.106079
4,30005,-240.906174
...,...,...
9994,39995,1121.636353
9995,39996,211.961594
9996,39997,585.485718
9997,39998,228.540176


In [29]:

output.to_csv('submission.csv', index=False)
print("done")

done
